# 04 - BM25 + Query Expansion

Notebook này xây dựng phương pháp truy xuất **BM25 + Query Expansion**.

Luồng chính:

```text
news_processed.pkl
→ tokens
→ BM25 Index thủ công
→ Search lần 1
→ Query Expansion từ top documents
→ Search lần 2
→ Top-k kết quả
```

## Cell 1 - Import thư viện

Cell này import các thư viện cần thiết để:
- Đọc dữ liệu đã xử lý.
- Tự xây BM25 index.
- Lưu index/model.

In [1]:
import os
import math
import pickle
import pandas as pd
import numpy as np

from collections import Counter, defaultdict

## Cell 2 - Import hàm xử lý query dùng chung

Cả 3 phương pháp dùng chung `preprocess_query()` trong `src/preprocess.py`.

In [2]:
import sys

CURRENT_DIR = os.getcwd()

if os.path.basename(CURRENT_DIR) == "notebooks":
    PROJECT_ROOT = os.path.abspath("..")
else:
    PROJECT_ROOT = CURRENT_DIR

if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

from src.preprocess import preprocess_query

## Cell 3 - Khai báo đường dẫn

Cell này khai báo:
- File dữ liệu đã xử lý.
- Folder lưu index/model của phương pháp BM25 + Query Expansion.

In [3]:
DATA_PATH = "../data/processed/news_processed.pkl"

MODEL_DIR = "../models/bm25_query_expansion"
os.makedirs(MODEL_DIR, exist_ok=True)

## Cell 4 - Load dữ liệu đã xử lý

Dữ liệu đầu vào là `news_processed.pkl`, dùng chung với các phương pháp khác.

In [4]:
df = pd.read_pickle(DATA_PATH)

print("Shape:", df.shape)
print(df.columns.tolist())

df[["doc_id", "title", "topic", "source", "processed_text"]].head()

Shape: (168169, 11)
['id', 'title', 'content', 'topic', 'source', 'url', 'raw_text', 'processed_text', 'tokens', 'token_count', 'doc_id']


,doc_id,title,topic,source,processed_text
0,0,"Tên cướp tiệm vàng tại Huế là đại uý công an, ...",Pháp luật,docbao.vn,tên cướp tiệm vàng huế đại_úy công_an công_tác...
1,1,"Bỏ qua mạng 5G, Nga tiến thẳng từ 4G lên 6G",Sống kết nối,vtc.vn,bỏ_qua mạng 5 g nga tiến thẳng 4 g lên 6 g bỏ_...
2,2,Địa phương nào đứng đầu cả nước tổng điểm 3 mô...,Giáo dục,thanhnien.vn,địa_phương nào đứng đầu cả nước tổng_điểm 3 mô...
3,3,Người chết trong mưa lũ 'nghìn năm có một' ở M...,Thế giới,vnexpress,người chết mưa_lũ nghìn năm mỹ tăng lên 28 ngư...
4,4,"Hải Phòng: Hình ảnh xe ""điên"" gây tai nạn liên...",Thời sự - Xã hội,soha,hải_phòng hình_ảnh xe điên gây tai_nạn liên_ho...


## Cell 5 - Lấy tokenized corpus

BM25 làm việc trực tiếp với danh sách token. Nếu cột `tokens` có sẵn thì dùng `tokens`, nếu không thì tách từ `processed_text`.

In [5]:
if "tokens" in df.columns:
    tokenized_corpus = df["tokens"].tolist()
else:
    tokenized_corpus = df["processed_text"].fillna("").astype(str).apply(lambda x: x.split()).tolist()

N = len(tokenized_corpus)

print("Số documents:", N)
print(tokenized_corpus[0][:50])

Số documents: 168169
['tên', 'cướp', 'tiệm', 'vàng', 'huế', 'đại_úy', 'công_an', 'công_tác', 'trại_giam', 'tên', 'cướp', 'tiệm', 'vàng', 'huế', 'đại_úy', 'công_an', 'công_tác', 'trại_giam', 'chiều', '31', '7', 'công_an', 'tỉnh', 'thừa', 'thiên_huế', 'thông_tin', 'ban_đầu', 'vụ', 'nổ_súng', 'cướp', 'tiệm', 'vàng', 'chợ', 'đông', 'ba', 'nằm', 'đường', 'trần_hưng_đạo', 'tp', 'huế', 'tỉnh', 'thừa', 'thiên_huế', 'thông_sài', 'gòn', 'giải_phóng', 'khoảng', '12', 'h30', 'ngày']


## Cell 6 - Tính độ dài document

BM25 cần:
- Độ dài từng document.
- Độ dài trung bình của toàn bộ corpus.

In [6]:
doc_lengths = np.array([len(tokens) for tokens in tokenized_corpus], dtype=np.float32)
avg_doc_length = doc_lengths.mean()

print("Average document length:", avg_doc_length)

Average document length: 341.432


## Cell 7 - Xây inverted index cho BM25

Inverted index có dạng:

```text
token → {doc_id: term_frequency}
```

Cấu trúc này giúp BM25 chỉ tính điểm trên các document chứa token trong query.

In [7]:
inverted_index = defaultdict(dict)
df_counter = Counter()

for doc_idx, tokens in enumerate(tokenized_corpus):
    token_counts = Counter(tokens)

    for token, tf in token_counts.items():
        inverted_index[token][doc_idx] = tf

    df_counter.update(token_counts.keys())

print("Số token trong inverted index:", len(inverted_index))

Số token trong inverted index: 515623


## Cell 8 - Tính IDF cho BM25

Công thức IDF của BM25:

```text
IDF(t) = log(1 + (N - DF(t) + 0.5) / (DF(t) + 0.5))
```

In [8]:
bm25_idf = {}

for token, df_t in df_counter.items():
    bm25_idf[token] = math.log(1 + (N - df_t + 0.5) / (df_t + 0.5))

print("Số token có IDF:", len(bm25_idf))

Số token có IDF: 515623


## Cell 9 - Tạo metadata

Metadata dùng để hiển thị kết quả truy xuất.

In [9]:
metadata = df[[
    "doc_id",
    "id",
    "title",
    "topic",
    "source",
    "url"
]].copy()

metadata.head()

,doc_id,id,title,topic,source,url
0,0,218270,"Tên cướp tiệm vàng tại Huế là đại uý công an, ...",Pháp luật,docbao.vn,https://docbao.vn/phap-luat/ten-cuop-tiem-vang...
1,1,218269,"Bỏ qua mạng 5G, Nga tiến thẳng từ 4G lên 6G",Sống kết nối,vtc.vn,https://vtc.vn/bo-qua-mang-5g-nga-tien-thang-t...
2,2,218268,Địa phương nào đứng đầu cả nước tổng điểm 3 mô...,Giáo dục,thanhnien.vn,https://thanhnien.vn/dia-phuong-nao-dung-dau-c...
3,3,218267,Người chết trong mưa lũ 'nghìn năm có một' ở M...,Thế giới,vnexpress,https://vnexpress.net/nguoi-chet-trong-mua-lu-...
4,4,218266,"Hải Phòng: Hình ảnh xe ""điên"" gây tai nạn liên...",Thời sự - Xã hội,soha,https://soha.vn/hai-phong-hinh-anh-xe-dien-gay...


## Cell 10 - Lưu BM25 index

Cell này lưu các thành phần cần thiết để search lại:
- `inverted_index.pkl`
- `bm25_idf.pkl`
- `doc_lengths.pkl`
- `metadata.pkl`

In [10]:
with open(f"{MODEL_DIR}/inverted_index.pkl", "wb") as f:
    pickle.dump(dict(inverted_index), f)

with open(f"{MODEL_DIR}/bm25_idf.pkl", "wb") as f:
    pickle.dump(bm25_idf, f)

with open(f"{MODEL_DIR}/doc_lengths.pkl", "wb") as f:
    pickle.dump(doc_lengths, f)

with open(f"{MODEL_DIR}/metadata.pkl", "wb") as f:
    pickle.dump(metadata, f)

with open(f"{MODEL_DIR}/avg_doc_length.pkl", "wb") as f:
    pickle.dump(avg_doc_length, f)

print("Đã lưu BM25 index vào:", MODEL_DIR)

Đã lưu BM25 index vào: ../models/bm25_query_expansion


## Cell 11 - Hàm tính điểm BM25

BM25 tính điểm dựa trên:
- TF của token trong document.
- IDF của token.
- Độ dài document.
- Độ dài trung bình của corpus.

In [11]:
def bm25_score(query_tokens, k1=1.5, b=0.75):
    scores = defaultdict(float)

    for token in query_tokens:
        if token not in inverted_index:
            continue

        idf = bm25_idf.get(token, 0.0)
        posting = inverted_index[token]

        for doc_idx, tf in posting.items():
            dl = doc_lengths[doc_idx]

            numerator = tf * (k1 + 1)
            denominator = tf + k1 * (1 - b + b * dl / avg_doc_length)

            scores[doc_idx] += idf * numerator / denominator

    return scores

## Cell 12 - Hàm search BM25 cơ bản

Hàm này:
1. Xử lý query.
2. Tính BM25 score.
3. Trả về top-k document có điểm cao nhất.

In [12]:
def search_bm25(query, top_k=10):
    query_processed, query_tokens = preprocess_query(query)

    scores_dict = bm25_score(query_tokens)

    if len(scores_dict) == 0:
        return pd.DataFrame(columns=list(metadata.columns) + ["score", "query_processed"])

    ranked = sorted(scores_dict.items(), key=lambda x: x[1], reverse=True)[:top_k]

    top_indices = [doc_idx for doc_idx, score in ranked]
    top_scores = [score for doc_idx, score in ranked]

    results = metadata.iloc[top_indices].copy()
    results["score"] = top_scores
    results["query_processed"] = query_processed

    return results

## Cell 13 - Hàm lấy từ mở rộng query

Query Expansion lấy thêm các token nổi bật từ top document của lần search đầu tiên.

Ở đây dùng cách đơn giản:
- Lấy top-k document đầu.
- Đếm tần suất token trong các document đó.
- Loại token đã có trong query.
- Lấy `expand_n` token phổ biến nhất để mở rộng query.

In [13]:
def get_expansion_terms(query_tokens, top_doc_indices, expand_n=5):
    query_token_set = set(query_tokens)

    token_counter = Counter()

    for doc_idx in top_doc_indices:
        token_counter.update(tokenized_corpus[doc_idx])

    expansion_terms = []

    for token, count in token_counter.most_common():
        if token not in query_token_set and token in bm25_idf:
            expansion_terms.append(token)

        if len(expansion_terms) >= expand_n:
            break

    return expansion_terms

## Cell 14 - Hàm search BM25 + Query Expansion

Quy trình:
1. Search BM25 lần 1.
2. Lấy top document làm nguồn mở rộng query.
3. Thêm từ mở rộng vào query.
4. Search BM25 lần 2.

In [14]:
def search_bm25_query_expansion(
    query,
    top_k=10,
    feedback_pool=10,
    expand_n=5
):
    query_processed, query_tokens = preprocess_query(query)

    first_scores = bm25_score(query_tokens)

    if len(first_scores) == 0:
        return pd.DataFrame(columns=list(metadata.columns) + ["score", "query_processed", "expanded_query"])

    first_ranked = sorted(first_scores.items(), key=lambda x: x[1], reverse=True)[:feedback_pool]
    top_doc_indices = [doc_idx for doc_idx, score in first_ranked]

    expansion_terms = get_expansion_terms(
        query_tokens=query_tokens,
        top_doc_indices=top_doc_indices,
        expand_n=expand_n
    )

    expanded_query_tokens = query_tokens + expansion_terms
    expanded_query = " ".join(expanded_query_tokens)

    second_scores = bm25_score(expanded_query_tokens)
    second_ranked = sorted(second_scores.items(), key=lambda x: x[1], reverse=True)[:top_k]

    top_indices = [doc_idx for doc_idx, score in second_ranked]
    top_scores = [score for doc_idx, score in second_ranked]

    results = metadata.iloc[top_indices].copy()
    results["score"] = top_scores
    results["query_processed"] = query_processed
    results["expanded_query"] = expanded_query

    return results

## Cell 15 - Test BM25 cơ bản

Cell này chạy thử BM25 chưa có Query Expansion.

In [15]:
query = "cầu thủ Quang Hải"

results = search_bm25(query, top_k=10)

print("Query gốc:", query)
print("Query sau xử lý:", results["query_processed"].iloc[0] if len(results) > 0 else "")

results[[
    "doc_id",
    "title",
    "topic",
    "source",
    "url",
    "score"
]]

Query gốc: cầu thủ Quang Hải
Query sau xử lý: cầu_thủ quang_hải


,doc_id,title,topic,source,url,score
120422,120422,"Báo Pháp: ""Sự vô danh của Quang Hải nhiều ngườ...",Thể thao,dantri,https://dantri.com.vn/the-thao/bao-phap-su-vo-...,16.214828
119849,119849,"Báo Pháp: ""Sự vô danh của Quang Hải khiến nhiề...",Thể thao,dantri,https://dantri.com.vn/the-thao/bao-phap-su-vo-...,16.202908
118562,118562,Hé lộ thời điểm Pau FC ra mắt tân binh Quang Hải,Thể thao,dantri,https://dantri.com.vn/the-thao/he-lo-thoi-diem...,16.006174
121636,121636,"PV Pháp: ""Cầu thủ châu Á ít được tin dùng ở Ph...",,soha,https://soha.vn/pv-phap-cau-thu-chau-a-it-duoc...,15.967932
134733,134733,"Chỉ ra những thách thức, báo Pháp nói Quang Hả...",Thể thao,baoquocte,https://baoquocte.vn/chi-ra-nhung-thach-thuc-b...,15.926889
134983,134983,Quang Hải đã hoàn tất thủ tục xin visa lao độn...,Thể thao,baoquocte,https://baoquocte.vn/quang-hai-da-hoan-tat-thu...,15.926889
114988,114988,"Thi đấu cho Pau FC, Quang Hải được báo Trung Q...",Thể thao,baoquocte,https://baoquocte.vn/thi-dau-cho-pau-fc-quang-...,15.902624
148844,148844,Cầu thủ Quang Hải bị va chạm giao thông trên đ...,Xã hội,dantri,https://dantri.com.vn/xa-hoi/cau-thu-quang-hai...,15.898200
13504,13504,Bình tĩnh chờ đợi Quang Hải - Tuổi Trẻ Online,Thể thao,tuoitre.vn,https://tuoitre.vn/binh-tinh-cho-doi-quang-hai...,15.884319
113684,113684,"HLV Pau FC thừa nhận một điều, so sánh Quang H...",Thể thao,dantri,https://dantri.com.vn/the-thao/hlv-pau-fc-thua...,15.857538


## Cell 16 - Test BM25 + Query Expansion

Cell này chạy thử BM25 sau khi mở rộng query.

In [16]:
query = "cầu thủ Quang Hải"

results = search_bm25_query_expansion(query, top_k=10, feedback_pool=10, expand_n=5)

print("Query gốc:", query)

if len(results) > 0:
    print("Query sau xử lý:", results["query_processed"].iloc[0])
    print("Query mở rộng:", results["expanded_query"].iloc[0])

results[[
    "doc_id",
    "title",
    "topic",
    "source",
    "url",
    "score"
]]

Query gốc: cầu thủ Quang Hải
Query sau xử lý: cầu_thủ quang_hải
Query mở rộng: cầu_thủ quang_hải việt_nam fc pháp người pau


,doc_id,title,topic,source,url,score
118562,118562,Hé lộ thời điểm Pau FC ra mắt tân binh Quang Hải,Thể thao,dantri,https://dantri.com.vn/the-thao/he-lo-thoi-diem...,44.548882
114575,114575,"Báo Pháp gọi Quang Hải là 'thần đồng', kỳ vọng...",,thethaovanhoa,https://thethaovanhoa.vn/bong-da-viet-nam/bao-...,44.342831
129403,129403,Báo thể thao số một nước Pháp khẳng định bến đ...,Thể thao,dantri,https://dantri.com.vn/the-thao/bao-the-thao-so...,44.196880
127544,127544,"Quang Hải: Báo Pháp chào mừng, fanpage CLB Pau...",Thể thao,baoquocte,https://baoquocte.vn/quang-hai-bao-phap-chao-m...,43.938385
130041,130041,Báo Pháp: 'Pau FC và Quang Hải đang đàm phán',Thể thao,zingnews,https://zingnews.vn/bao-phap-pau-fc-va-quang-h...,43.878288
64000,64000,"Pau FC ""trợ giúp"" Quang Hải sớm hòa nhập với đ...",Thể thao,dantri,https://dantri.com.vn/the-thao/pau-fc-tro-giup...,43.857986
115921,115921,"Báo Trung Quốc: ""Pau FC chưa phải điểm dừng cu...",,soha,https://soha.vn/bao-trung-quoc-pau-fc-chua-pha...,43.834900
17013,17013,Báo Pháp nhận xét bất ngờ về Quang Hải,Thể thao,vietnamnet.vn,https://vietnamnet.vn/bao-phap-nhan-xet-bat-ng...,43.572556
129229,129229,"Quang Hải chia tay gia đình, chuẩn bị sang Phá...",Thể thao,laodong,https://laodong.vn/bong-da/quang-hai-chia-tay-...,43.563835
63835,63835,Pau FC có động thái giúp Quang Hải sớm hòa nhậ...,Thể thao,dantri,https://dantri.com.vn/the-thao/pau-fc-co-dong-...,43.493088


## Cell 17 - Hiển thị kết quả dạng dễ đọc

Cell này in kết quả theo từng rank để dễ kiểm tra nội dung bài báo.

In [17]:
def show_results(results):
    for rank, (_, row) in enumerate(results.iterrows(), start=1):
        print("=" * 100)
        print(f"Rank {rank}")
        print("Score :", round(row["score"], 4))
        print("Title :", row["title"])
        print("Topic :", row["topic"])
        print("Source:", row["source"])
        print("URL   :", row["url"])
        if "expanded_query" in results.columns:
            print("Expanded query:", row.get("expanded_query", ""))
        print("Content preview:")
        print(str(row["content"])[:500])

In [18]:
show_results(results)

Rank 1
Score : 44.5489
Title : Hé lộ thời điểm Pau FC ra mắt tân binh Quang Hải
Topic : Thể thao
Source: dantri
URL   : https://dantri.com.vn/the-thao/he-lo-thoi-diem-pau-fc-ra-mat-tan-binh-quang-hai-20220629204620978.htm
Expanded query: cầu_thủ quang_hải việt_nam fc pháp người pau
Content preview:


KeyError: 'content'